# Reproduce KALIA's optimizer ablation (free Kaggle GPU)

Trains the same 30M-parameter model two ways — AdamW vs Muon+QK-Norm+soft-cap — on a TinyStories slice, then prints the validation-loss comparison. About 1 hour on GPU T4 x2.

This notebook exists so anyone can verify the project's central claim on their own compute: on equal tokens and equal seeds, the Muon-based recipe reaches a lower validation loss than AdamW.

**Settings:** Accelerator: GPU T4 x2 · Internet: On.

In [ ]:
!pip install -q datasets tiktoken pyyaml

In [ ]:
GITHUB_URL = "https://github.com/CHANGE_ME/kalia.git"  # <-- the public KALIA repository

!git clone {GITHUB_URL} || (cd kalia && git pull)
%cd kalia

In [ ]:
import os

import numpy as np
import tiktoken
from datasets import load_dataset

os.makedirs("data", exist_ok=True)
encoder = tiktoken.get_encoding("gpt2")
target_tokens = 82_000_000
written = 0
buffer = []

with open("data/all.bin", "wb") as fh:
    for row in load_dataset("roneneldan/TinyStories", split="train", streaming=True):
        ids = encoder.encode_ordinary(row["text"])
        ids.append(encoder.eot_token)
        buffer.extend(ids)
        if len(buffer) >= 1_000_000:
            np.array(buffer, dtype=np.uint16).tofile(fh)
            written += len(buffer)
            buffer = []
            if written >= target_tokens:
                break
    if buffer:
        np.array(buffer, dtype=np.uint16).tofile(fh)
        written += len(buffer)

print("tokenized:", written)
arr = np.memmap("data/all.bin", dtype=np.uint16, mode="r")
arr[: written - 2_000_000].tofile("data/train.bin")
arr[written - 2_000_000 : written].tofile("data/val.bin")
print("train:", (written - 2_000_000), "| val: 2000000")

In [ ]:
!torchrun --nproc_per_node=2 --standalone train.py --config configs/micro-base.yaml --data-dir data --out-dir out/base --max-steps 500
!torchrun --nproc_per_node=2 --standalone train.py --config configs/micro-muonplus-qk.yaml --data-dir data --out-dir out/upgraded --max-steps 500

In [ ]:
import csv

def final_val(path):
    rows = list(csv.reader(open(path)))[1:]
    return rows[-1] if rows else ["n/a", "n/a"]

print("AdamW baseline :", final_val("out/base/val_log.csv"))
print("Muon+ recipe   :", final_val("out/upgraded/val_log.csv"))
print()
print("Expected: the Muon+ recipe reaches a lower validation loss at equal tokens.")

Full-scale results, the incident journal and the decision ledger live in the repository under `docs/`. The ablation reported here (30M params, 50M tokens, step 700) produced 3.8041 (AdamW) → 3.5937 (Muon) → 3.5103 (Muon+QK-Norm+soft-cap) in the project's own runs.